# 0. Predicting `Block Position Cycle` Using Machine Learning

*Block Cycle Prediction and Comparison of Seven Scheduling Results*

> 공개된 조선 Block 데이터를 이용해 **스케줄링 이전의 기준 작업기간(Block Position Cycle)**을 Block 고유 특성으로 예측하고, 동일한 입력조건에서 계산된 7개 스케줄링 방법의 Position 배정·계획일정·Delay 결과를 비교하는 프로젝트

이 프로젝트는 머신러닝 예측과 스케줄링 비교를 하나의 인과모델로 섞지 않고 두 단계로 분리한다.

```text
① Block 고유 특성 → Block Position Cycle 예측

② Block / Position / Initial / Cycle → Scheduling Method
   → Position 배정 / Planned Start·Finish / Delay
```


## 0.1 프로젝트를 시작한 이유

### 0.1.1 개인 경험에서 출발한 문제의식

현재 교육과정은 기술 자체를 배우는 데서 끝나는 것이 아니라, 현장에서 실제로 사용할 수 있는 해결책을 만드는 과정이라고 생각했다. 비전공자로서 개발 이론과 조선 실무 지식은 부족했지만, 건축 전공과 설계사무소 경험에서 발견한 문제를 조선업과 연결해 보았다.

건축 설계 실무에서 디지털 도면과 실제 현장 사이의 차이, 반복적인 설계변경과 수정 업무를 경험했다. 조선 역시 다수 조직과 복잡한 공정이 동시에 움직이는 산업이라는 점에서 비슷한 문제가 존재한다고 판단했다.

### 0.1.2 건축과 조선이 공유하는 어려움

- 디지털 도면과 실제 작업 현장 사이의 괴리
- 잦은 설계변경과 프로젝트 규모에 비례해 늘어나는 수정 업무
- 복잡한 공정과 다수 조직이 얽힌 일회성 프로젝트 중심의 분절된 협업구조(one-off / fragmented organization)

### 0.1.3 검토한 아이디어와 최종 선택

- 로보틱스 측량 시스템: 현장 측정 자동화
- BIM을 조선에 적용한 SBIM(Shipbuilding Information Modeling)
- 스마트 공정계산 및 협력업체 코디네이팅 시스템

이 가운데 교육과정에서 배운 데이터 전처리, 회귀분석, 모델 평가를 가장 직접적으로 활용할 수 있는 주제가 **작업기간 예측**이라고 판단했다. 처음에는 예측 결과를 일정으로 시각화하고, 장기적으로는 현장 변화에 따라 일정을 다시 추천하는 시스템까지 확장하는 것을 목표로 삼았다.

## 0.2 참고자료 및 데이터셋

### 0.2.1 참고자료

#### 0.2.1.1 분석 방법론

- **수업 강의자료 및 회귀분석 예제** (삼성중공업 DT 5기, 2026)
  - 회귀분석, 모델 평가 및 분석 코드 구성에 활용

#### 0.2.1.2 Scheduling Process Case Study

- **「조선 산업의 블록 적치장 운영계획 휴리스틱 알고리즘」** (손정열·서흥원·하병현, 2014)
  - 조선 Block의 적치·Position 제약 및 휴리스틱 Scheduling Process 이해

- **「조선소 선행의장 공정의 스케줄링 최적화를 위한 알고리즘 비교 분석 연구」** (김기현·나영진·박세호·이동녕·장종환·임민중, 연도 미상)
  - 조선 공정의 제약조건·목적함수 및 Scheduling Algorithm 비교 사례 검토

- **「조선소 블록 조립 스케줄링을 위한 자연 모사 알고리즘 비교 연구」** (박세호·김기현·나영진·이동녕·장종환·임민중, 연도 미상)
  - 조선 Block Scheduling의 알고리즘별 성능 비교 사례 검토

#### 0.2.1.3 검토 데이터셋

- **A Ship-Construction Dataset for Resource Leveling Optimization in Large Project Management Problems** (저자·연도 확인 필요)
  - 프로젝트 초기 데이터셋 후보로 검토

#### 0.2.1.4 최종 분석 데이터셋

- **Data of Ship Block Scheduling in English** (Shaozi Chen, 2026)
  - 본 프로젝트의 Block / Position / Initial 및 7개 Scheduling Result 분석에 사용

#### 0.2.1.5 핵심 선행연구

- **Knowledge-Based Curved Block Construction Scheduling and Application in Shipbuilding** (Jiang Zuhua, Zhou Hongming, Tao Ningrong & Li Baihe, 2024)
  - 최종 데이터셋과 관련된 연구계보 검토
  - Block Cycle, Position, Initial State 및 Scheduling Process의 관계 해석에 활용

- **A Study on the Man-Hour Prediction System for Shipbuilding** (Hur Minhoe, Lee Seung-kyung, Kim Bongseok, Cho Sungzoon, Lee Dongha & Lee Daehyung, 2015)
  - 위 핵심 선행연구의 주요 참조문헌
  - 조선업의 사전 Man-hour 산정 및 생산단계별 재예측 구조 해석에 활용


### 0.2.2 데이터셋 선정 이유

- 초기에는 건축공정 데이터로 회귀분석을 시험하려 했으나, 자료 조사 과정에서 조선 블록 스케줄링 관련 공개 데이터셋을 발견하여 조선 데이터 활용 가능성을 검토하였다.
- 'A Ship-Construction Dataset for Resource Leveling Optimization in Large Project Management Problems'은 제공 데이터가 인코딩되어 있고, 컬럼 정의가 제공되지 않아 분석에 활용하기 어려웠다.
- 'Data of Ship Block Scheduling in English'에 첨부된 Figure6&7_raw_data는 조선 블록 스케줄링을 재현할 수 있는 원시 데이터를 제공한다. 또한 블록과 작업 위치의 속성, 초기 배정 상태 및 여러 스케줄링 알고리즘의 최신 분석 결과(2026.03)가 함께 수록되어 있어 공정시간에 영향을 미치는 변수를 분석하고 회귀모델을 구성하기에 적합하다고 판단하였다. 


### 0.2.3 데이터셋 구성

| 자료 | 주요 내용 | 이 프로젝트에서의 역할 |
|---|---|---|
| `block table` | 블록 크기, 중량, 선박 번호, Block Position Cycle | 예측 대상과 Block 특성 |
| `position table` | 작업 위치 크기, 인양 능력, 높이 제한, 작업조직 | Position 특성 |
| `initial table` | 스케줄링 시작 시점의 초기 야드 상태 | 초기조건 확인용 |
| `Result` | DDQN, EDDQN, 5개 휴리스틱의 스케줄링 결과 | Block–Position 연결표 |
| `README.md` | 테이블 및 일부 컬럼 설명 | 데이터 구조 해석 근거 |

### 0.2.4 데이터셋의 주요 전제

- Block과 Position의 투영형상을 최소 경계 직사각형으로 단순화한다.
- 한 Position은 동시에 한 Block만 처리한다.
- 노동력은 충분하며 결근과 같은 이상 상황은 고려하지 않는다.
- 서로 다른 크기의 Block은 적절한 Position을 찾을 수 있다고 가정한다.
- 하나의 Block은 하나의 Position에서 작업을 완료한다.

## 0.3 종속변수와 데이터 생성 순서

프로젝트를 진행하면서 데이터의 의미와 생성 순서를 두 차례 다시 검토했다.

### 0.3.1 첫 번째 재검토 — Cycle의 의미

초기에는 `Block Position Cycle`을 작업 완료 후 측정되는 실제 처리기간으로 해석하였다. 그러나 원본 구조를 다시 확인한 결과 Cycle은 알고리즘별 Result가 아니라 **스케줄링 입력인 Block Table에 미리 존재**한다.

따라서 본 프로젝트에서는 `Block Position Cycle`을 **스케줄링 이전에 Block별로 주어진 기준 처리기간**으로 정의한다. 정확한 산정식과 원천은 공개자료에서 확인되지 않았으므로, 실제 작업 후 측정된 실적 공기와 동일하게 해석하지 않는다.

### 0.3.2 두 번째 재검토 — 선택 Position의 역할

Position Information Table은 스케줄링 전에 존재하는 후보 위치 정보다. 그러나 특정 Block에 연결된 **선택 Position**은 Scheduling Method가 후보 중 하나를 선택한 결과다.

```text
Position 후보 정보                         선택 Position
스케줄링 이전의 입력조건          →       Scheduling Result
모든 방법에 공통으로 제공                 방법에 따라 달라짐
```

즉 Position 후보 정보와 선택 Position 결과를 같은 변수로 취급하면 안 된다. 데이터를 Join할 수 있다는 사실만으로 해당 변수를 사전 예측모델에 사용할 수 있는 것도 아니다.

### 0.3.3 최종 분석 구조

- **Cycle 예측**: 스케줄링 결과보다 먼저 알 수 있는 Block 고유 특성만 입력한다.
- **탐색적 진단**: DDQN이 선택한 Position과 Initial 정보를 붙인 Set B/C는 사후 정보 추가 시 성능 변화를 확인하는 용도로만 사용한다.
- **Scheduling 비교**: 7개 방법이 만든 Position 배정, Planned Start/Finish, Makespan과 Delay를 결과변수로 비교한다.

> 독립변수는 단순히 연결 가능한 컬럼이 아니라, 예측하려는 시점에 실제로 알 수 있는 값이어야 한다.


## 0.4 프로젝트 목표

1. **Block 고유 특성만으로 사전 Block Position Cycle을 어느 정도 예측할 수 있는가?**
2. DDQN이 선택한 Position·Initial 정보를 사후에 추가하면 통계적 예측 성능은 어떻게 변하는가?
3. 동일한 Block과 야드 조건에서 **7가지 Scheduling Method가 선택한 Position과 계획일정은 얼마나 다른가?**
4. 각 Result의 **Makespan과 Delay 지표**를 비교했을 때 방법별 일정 성과는 어떻게 다른가?

> Set A(Block)는 최종 예측모델이다. Set B/C는 데이터 생성 순서를 검토하기 위한 탐색적 비교이며, Position의 인과효과나 스케줄링 우열을 평가하지 않는다.


## 0.5 전체 분석 흐름

### 0.5.1 Master Table 생성 및 연결 검증
[1번 노트북 열기](<Predicting 'Block Position Cycle'_1_master table.ipynb>)

- Block–Result–Position–Initial 연결키와 입력층/결과층 검증
- DDQN 결과를 붙인 탐색용 Master Table 생성
- 선택 Position이 Block 고유 특성이 아니라 Scheduling Result임을 명시

### 0.5.2 데이터 전처리 및 EDA
[2번 노트북 열기](<Predicting 'Block Position Cycle'_2_preprocessing_eda.ipynb>)

- 결측값·이상치 검토와 Block/Position 파생변수 생성
- 최종 예측용 Set A와 탐색용 Set B/C 구성
- Cycle과 Block 고유 특성의 관계 및 사후 Position 정보의 분포 확인

### 0.5.3 회귀모델 구축 및 평가
[3번 노트북 열기](<Predicting 'Block Position Cycle'_3_modeling.ipynb>)

- Set A에서 Linear, Ridge, Decision Tree, Random Forest 비교
- Set A/B/C 성능 비교를 통한 데이터 생성 순서 진단
- **Set A를 최종 변수 세트로 확정**하고 튜닝·잔차·중요도 분석

### 0.5.4 7개 Scheduling Result 비교
[4번 노트북 열기](<Predicting 'Block Position Cycle'_4_scheduling_comparison.ipynb>)

- 방법별 Position 배정과 동일 Position 선택률 비교
- Planned Start/Finish와 Makespan 비교
- Total Delay, Delayed Blocks, Max Delay 비교
- Cycle 예측오차를 스케줄링 성과로 사용하지 않음

### 0.5.5 최종 결론
[5번 노트북 열기](<Predicting 'Block Position Cycle'_5_conclusion.ipynb>)

- 초기 가정, 논리 오류 발견과 분석구조 수정 과정 정리
- Cycle 예측과 Scheduling Result 비교 결과 종합
- 한계와 향후 과제 정리

### 0.5.6 개발 로드맵
[6번 노트북 열기](<Predicting 'Block Position Cycle'_6_development_roadmap.ipynb>)

- Set A Cycle 예측모델과 Scheduling Engine의 역할 분리
- 7개 Result 공간·시간 Viewer
- 실적 데이터 확보 후 동적 재예측·재스케줄링으로 확장
